# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [33]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
import gradio as gr
import json

In [13]:

MODEL_GPT = 'openai/gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [14]:
load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [15]:
ODEL_GPT = "openai/gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [26]:
football_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_live_tactics_and_stats",
            "parameters": {
                "type": "object",
                "properties": {
                    "team_name": {
                        "type": "string",
                        "description": "Name of the football club",
                    }
                },
                "required": ["team_name"],
            },
            "description": "Fetches current tactical setup, form, and stats for a given football team.",
        },
    }
]

In [ ]:
def execute_tool(tool_call):
    """Executes local tool logic when triggered by LLM."""
    fn_name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    print("Tool Called")

    if fn_name == "get_live_tactics_and_stats":
        team = args.get("team_name", "the team")
        # Mock database / API payload
        result = {
            "team": team,
            "formation": "4-3-3 High Press",
            "recent_form": "WWDLW",
            "top_scorer": "9 goals in 5 matches",
            "defensive_style": "High offside trap",
        }
        return json.dumps(result)
    return "{}"

In [36]:
def chat_with_model(user_message, model_choice, response_style):
    system_prompt = f"""
You are an expert football coach and tactical analyst. 
Explain every concept using vivid football metaphors.

INSTRUCTIONS:
1. Tone/Style: {response_style}
2. Explain complex ideas by comparing them directly to match-day scenarios.
3. Keep explanations clear, engaging, and easy to follow.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    # Handle Tool Calling (OpenRouter)
    if model_choice == "GPT (OpenRouter)":
        tool_check = client.chat.completions.create(
            model=MODEL_GPT, messages=messages, tools=football_tools
        )
        msg = tool_check.choices[0].message
        if msg.tool_calls:
            messages.append(msg)
            for t_call in msg.tool_calls:
                tool_result = execute_tool(t_call)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": t_call.id,
                        "content": tool_result,
                    }
                )

    # Initialize Text Stream
    if model_choice == "GPT (OpenRouter)":
        stream = client.chat.completions.create(
            model=MODEL_GPT, messages=messages, temperature=0.3, stream=True
        )
    elif model_choice == "LLaMA (Ollama)":
        import ollama
        stream = ollama.chat(model=MODEL_LLAMA, messages=messages, stream=True)

    # Stream Text to Gradio UI
    full_response = ""
    for chunk in stream:
        if hasattr(chunk, "choices"):
            content = chunk.choices[0].delta.content
        elif isinstance(chunk, dict):
            content = chunk.get("message", {}).get("content")
        else:
            content = None

        if content:
            full_response += content
            yield full_response

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ⚽ Tactical AI Coach")
    gr.Markdown("Ask any question and get it explained in football terms!")

    with gr.Row():
        with gr.Column(scale=2):
            user_input = gr.Textbox(
                label="Ask your question",
                lines=4,
                placeholder="e.g., How does load balancing work in software engineering?",
            )

            with gr.Row():
                model_dropdown = gr.Dropdown(
                    choices=["GPT (OpenRouter)", "LLaMA (Ollama)"],
                    value="GPT (OpenRouter)",
                    label="⚡ Engine",
                )

                response_style = gr.Dropdown(
                    choices=[
                        "Pure Football Analogy (100% Match terms)",
                        "Hybrid (Technical explanation + Football Metaphor)",
                        "Angry Locker Room Manager (Halftime Speech)",
                    ],
                    value="Pure Football Analogy (100% Match terms)",
                    label="🎙️ Commentary Style",
                )

            submit_btn = gr.Button("⚽ Explain Concept", variant="primary")

        with gr.Column(scale=2):
            text_output = gr.Markdown(label="Coach Response")

    submit_btn.click(
        fn=chat_with_model,
        inputs=[user_input, model_dropdown, response_style],
        outputs=text_output,
    )

demo.launch()